# Refine

> Postprocessing markdown files including fixing heading hierarchy and adding image descriptions

In [ ]:
#| default_exp refine

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Adding page numbers to headings for better navigation
3. Enriching figure references with descriptive text and creating a table of figures

[TBD]

In [ ]:
#| export
from mistocr.core import read_pgs
from re import sub, findall

# Dev only
import sys
sys.path.append('../../..')
from ctx_utils import *

In [ ]:
md = read_pgs('files/test/md_all/resnet')
md[:5000]

'# Deep Residual Learning for Image Recognition \n\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\\{kahe, v-xiangz, v-shren, jiansun\\}@microsoft.com\n\n\n#### Abstract\n\nDeeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers- $8 \\times$ deeper than VGG nets [41] but still having lower complexity. An ensemble of these residual nets achieves $3.57 \\%$ error on the ImageNet test set. This result won the 1st place on the ILSVRC 2015 classification

How to retrieve all md headings?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

To retrieve all markdown headings from your text, you can use a regular expression to match lines that start with `#` symbols. Here's a simple approach:

```python
import re
headings = re.findall(r'^#{1,6} .+$', md, re.MULTILINE)
```

This pattern matches lines starting with 1-6 `#` symbols followed by a space and some text.

Would you like to try this and see what headings you get from your `md` variable? Or would you prefer a different approach that gives you more information about each heading (like its level or position)?

In [ ]:
import re
hdgs = re.findall(r'^#{1,6} .+$', md, re.MULTILINE)
hdgs

['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']

The hierarchy of those headings looks broken. How to fix it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right! The hierarchy is broken - notice how it jumps from `#` (level 1) to `####` (level 4) for "Abstract", and later has `##` headings like "PASCAL VOC" and "MS COCO" that should probably be subsections.

To fix heading hierarchy, you need to decide on a strategy. Here are some common approaches:

1. **Normalize sequentially** - ensure headings increment by only 1 level at a time (e.g., # → ## → ### not # → ####)
2. **Use context clues** - look at the content/position to determine the proper level (e.g., "Abstract" is typically level 2)
3. **Fix specific patterns** - like ensuring appendix sections (A, B, C) are at the right level

Which approach makes sense for your use case? Or would you like to explore what the actual structure should be for this academic paper format?

Yes indeed ensuring that headings increment by only 1 level make sense. However though here the hierarchy (apart from the Abstract) is ok, I've noticed sometimes that the headings are wrong especially on long and complex documents. When available the numbering also helps fixing it. Don't you think that an LLM could easily fix it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Absolutely! An LLM would be excellent for this task. It could:

- Understand document structure semantically (e.g., "Abstract" typically comes after the title)
- Use the numbering in headings (like "3.1", "3.2") to determine proper nesting
- Handle edge cases that regex rules might miss
- Fix inconsistencies across long documents where patterns vary

You could give it the list of headings (or the full markdown) and ask it to return a corrected version with proper hierarchy.

Have you already set up an LLM API to use (like Claude, GPT, etc.)? Or would you like to think through what the prompt should look like for this task?

In [ ]:
from toolslm.md_hier import *

In [ ]:
hdgs = create_heading_dict(md)
hdgs

{'Deep Residual Learning for Image Recognition': {'Abstract': {},
  '1. Introduction': {},
  '2. Related Work': {},
  '3. Deep Residual Learning': {'3.1. Residual Learning': {},
   '3.2. Identity Mapping by Shortcuts': {},
   '3.3. Network Architectures': {},
   '3.4. Implementation': {}},
  '4. Experiments': {'4.1. ImageNet Classification': {},
   '4.2. CIFAR-10 and Analysis': {},
   '4.3. Object Detection on PASCAL and MS COCO': {}},
  'References': {},
  'A. Object Detection Baselines': {},
  'PASCAL VOC': {},
  'MS COCO': {},
  'B. Object Detection Improvements': {},
  'ImageNet Detection': {},
  'C. ImageNet Localization': {}}}

In [ ]:
print(hdgs['Deep Residual Learning for Image Recognition']['1. Introduction'].text)

## 1. Introduction

Deep convolutional neural networks [22, 21] have led to a series of breakthroughs for image classification [21, 50, 40]. Deep networks naturally integrate low/mid/highlevel features [50] and classifiers in an end-to-end multilayer fashion, and the "levels" of features can be enriched by the number of stacked layers (depth). Recent evidence $[41,44]$ reveals that network depth is of crucial importance, and the leading results $[41,44,13,16]$ on the challenging ImageNet dataset [36] all exploit "very deep" [41] models, with a depth of sixteen [41] to thirty [16]. Many other nontrivial visual recognition tasks $[8,12,7,32,27]$ have also

[^0]![img-0.jpeg](img-0.jpeg)

Figure 1. Training error (left) and test error (right) on CIFAR-10 with 20-layer and 56-layer "plain" networks. The deeper network has higher training error, and thus test error. Similar phenomena on ImageNet is presented in Fig. 4.
greatly benefited from very deep models.
Driven by the significance of de

In [ ]:
md_hier_py = r'''import re
from fastcore.utils import *
__all__ = ['create_heading_dict', 'HeadingDict']

class HeadingDict(dict):
    """A dictionary-like object that also stores the markdown text content."""
    def __init__(self, text="", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.text = text


def create_heading_dict(text, rm_fenced=True):
    "Create a nested dictionary structure from markdown headings."
    original_text = text
    original_lines = text.splitlines()
    
    # Use fenced-removed text only for finding headings
    text_for_headings = text
    if rm_fenced: text_for_headings = re.sub(r'```[\s\S]*?```', '', text)

    lines_for_headings = text_for_headings.splitlines()
    headings = []

    # Parse headings with their levels and line numbers
    for idx, line in enumerate(lines_for_headings):
        match = re.match(r'^(#{1,6})\s+\S.*', line)
        if match:
            level = len(match.group(1))
            title = line.strip('#').strip()
            headings.append({'level': level, 'title': title, 'line': idx})

    # Assign text content to each heading using original lines
    for i, h in enumerate(headings):
        start = h['line']
        # Find the end index: next heading of same or higher level
        for j in range(i + 1, len(headings)):
            if headings[j]['level'] <= h['level']:
                end = headings[j]['line']
                break
        else:
            end = len(original_lines)
        h['content'] = '\n'.join(original_lines[start:end]).strip()

    # Build the nested structure
    result = HeadingDict(original_text)
    stack = [result]
    stack_levels = [0]

    for h in headings:
        # Pop stack until we find the right parent level
        while len(stack) > 1 and stack_levels[-1] >= h['level']:
            stack.pop()
            stack_levels.pop()

        new_dict = HeadingDict(h['content'])
        stack[-1][h['title']] = new_dict
        stack.append(new_dict)
        stack_levels.append(h['level'])

    return result


if __name__=='__main__':
    md_content = """
# User

This is the User section.

## Tokens

Details about tokens.

### Value

The value of tokens.

Some more details.

## Settings

User settings information.

# Admin

Admin section.

## Users

Admin users management.
"""

    result = create_heading_dict(md_content)
    #for key, value in result.items(): print(f'Key: {key}\nValue:\n{value}\n{"-"*40}')

    def test_empty_content():
        md_content = "# Empty Heading"
        result = create_heading_dict(md_content)
        assert 'Empty Heading' in result
        assert result['Empty Heading'].text == '# Empty Heading'
        assert result.text == md_content

    def test_special_characters():
        md_content = "# Heading *With* Special _Characters_!\nContent under heading."
        result = create_heading_dict(md_content)
        assert 'Heading *With* Special _Characters_!' in result
        assert result['Heading *With* Special _Characters_!'].text == '# Heading *With* Special _Characters_!\nContent under heading.'
        assert result.text == md_content

    def test_duplicate_headings():
        md_content = "# Duplicate\n## Duplicate\n### Duplicate\nContent under duplicate headings."
        result = create_heading_dict(md_content)
        assert 'Duplicate' in result
        assert 'Duplicate' in result['Duplicate']
        assert 'Duplicate' in result['Duplicate']['Duplicate']
        assert result['Duplicate']['Duplicate']['Duplicate'].text == '### Duplicate\nContent under duplicate headings.'
        assert result.text == md_content

    def test_no_content():
        md_content = "# No Content Heading\n## Subheading"
        result = create_heading_dict(md_content)
        assert result['No Content Heading'].text == '# No Content Heading\n## Subheading'
        assert result['No Content Heading']['Subheading'].text == '## Subheading'
        assert result.text == md_content

    def test_different_levels():
        md_content = "### Level 3 Heading\nContent at level 3.\n# Level 1 Heading\nContent at level 1."
        result = create_heading_dict(md_content)
        assert 'Level 3 Heading' in result
        assert 'Level 1 Heading' in result
        assert result['Level 3 Heading'].text == '### Level 3 Heading\nContent at level 3.'
        assert result['Level 1 Heading'].text == '# Level 1 Heading\nContent at level 1.'
        assert result.text == md_content

    def test_parent_includes_subheadings():
        md_content = "# Parent\nParent content.\n## Child\nChild content.\n### Grandchild\nGrandchild content."
        result = create_heading_dict(md_content)
        assert result['Parent'].text == '# Parent\nParent content.\n## Child\nChild content.\n### Grandchild\nGrandchild content.'
        assert result['Parent']['Child'].text == '## Child\nChild content.\n### Grandchild\nGrandchild content.'
        assert result['Parent']['Child']['Grandchild'].text == '### Grandchild\nGrandchild content.'
        assert result.text == md_content

    def test_multiple_level2_siblings():
        md_content = "## Sib 1\n## Sib 2\n## Sib 3\n## Sib 4\n## Sib 5'"
        result = create_heading_dict(md_content)
        assert 'Sib 1' in result
        assert 'Sib 2' in result
        assert 'Sib 3' in result
        assert 'Sib 4' in result
        assert "Sib 5'" in result
        assert result.text == md_content

    def test_code_chunks_escaped():
        md_content = "# Parent\nParent content.\n## Child\nChild content.\n```python\n# Code comment\nprint('Hello, world!')\n```"
        result = create_heading_dict(md_content)
        assert 'Code comment' not in str(result)
        assert result.text == md_content

    test_empty_content()
    test_special_characters()
    test_duplicate_headings()
    test_no_content()
    test_different_levels()
    test_parent_includes_subheadings()
    test_multiple_level2_siblings()
    test_code_chunks_escaped()
    print('tests passed')

    def test_nested_headings():
        md_content = "# Parent\nParent content.\n## Child\nChild content.\n### Grandchild\nGrandchild content."
        result = create_heading_dict(md_content)
        assert 'Child' in result['Parent']
        assert 'Grandchild' in result['Parent']['Child']

    def test_code_chunks_escaped():
        md_content = "# Parent\nParent content.\n## Child\nChild content.\n```python\n# Code comment\nprint('Hello, world!')\n```"
        result = create_heading_dict(md_content)
        assert 'Code comment' not in result

    def test_fenced_blocks_preserved_in_text():
        md_content = """# Section
Content before code.

```python
# This heading should be ignored for structure
def hello():
    print("Hello, world!")
```

More content after code."""
        result = create_heading_dict(md_content)
        # Fenced code should be preserved in text content
        assert '```python' in result['Section'].text
        assert 'def hello():' in result['Section'].text
        assert '```' in result['Section'].text
        # But headings inside fenced blocks should not create structure
        assert 'This heading should be ignored for structure' not in result['Section']

    test_nested_headings()
    test_code_chunks_escaped()
    test_fenced_blocks_preserved_in_text()

    def test_multiple_h1s():
        md_content = "# First H1\n# Second H1\n# Third H1"
        result = create_heading_dict(md_content)
        assert 'First H1' in result
        assert 'Second H1' in result
        assert 'Third H1' in result
        assert result['First H1'] == {}
        assert result['Second H1'] == {}
        assert result['Third H1'] == {}

    def test_skip_levels_down():
        md_content = "# Root\n## Level2\n#### Level4"
        result = create_heading_dict(md_content)
        assert 'Root' in result
        assert 'Level2' in result['Root']
        assert 'Level4' in result['Root']['Level2']

    def test_skip_levels_up():
        md_content = "# Root\n#### Deep\n## Back to 2"
        result = create_heading_dict(md_content)
        assert 'Root' in result
        assert 'Deep' in result['Root']
        assert 'Back to 2' in result['Root']
        assert result['Root']['Deep'] == {}
        assert result['Root']['Back to 2'] == {}

    def test_non_h1_start():
        md_content = "### Starting at 3\n## Going to 2\n# Finally 1"
        result = create_heading_dict(md_content)
        assert 'Starting at 3' in result
        assert 'Going to 2' in result
        assert 'Finally 1' in result

    test_multiple_h1s()
    test_skip_levels_down()
    test_skip_levels_up()
    test_non_h1_start()

    # Edge case tests
    def test_empty_input():
        result = create_heading_dict("")
        assert result == {}
        assert result.text == ""

    def test_whitespace_only():
        result = create_heading_dict("   \n\t  \n   ")
        assert result == {}
        assert result.text == "   \n\t  \n   "

    def test_malformed_headings():
        # Too many #s (matches max 6)
        md_content = "####### Too Many\nContent"
        result = create_heading_dict(md_content)
        assert 'Too Many' not in result
        assert result.text == md_content

    def test_unicode_and_emojis():
        # Unicode characters
        md_content = "# Café & Naïve\nContent with unicode\n## 中文标题\nChinese content"
        result = create_heading_dict(md_content)
        assert 'Café & Naïve' in result
        assert '中文标题' in result['Café & Naïve']
        assert result.text == md_content

        # Emojis
        md_content = "# 🚀 Rocket Heading\nRocket content\n## 💻 Computer\nComputer content"
        result = create_heading_dict(md_content)
        assert '🚀 Rocket Heading' in result
        assert '💻 Computer' in result['🚀 Rocket Heading']
        assert result.text == md_content

    test_empty_input()
    test_whitespace_only()
    test_malformed_headings()
    test_unicode_and_emojis()
    print('tests passed')
'''

The regexp used to detect headings is a bit naive and might not work, for instance if there is embedded python code (commented) in a paper for instance. 
I particularly like the $`md_hier_py` module as it addresses these edge case for instance. I am wondering if we could leverage this module to fix the headings. It seems also that it fixed the "abstract" heading issue (why/how)?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great observation! Looking at the `create_heading_dict` function, it handles the code block issue through this line:

```python
if rm_fenced: text_for_headings = re.sub(r'```[\s\S]*?```', '', text)
```

It removes fenced code blocks before looking for headings, so commented Python headers won't be mistakenly parsed.

Regarding the "Abstract" - it didn't actually *fix* the level (it's still `####` in the markdown). The function just built the nested dictionary structure based on whatever levels it found. So "Abstract" at level 4 became nested under the title at level 1, which happens to look reasonable structurally, but the underlying markdown is still wrong.

To actually *fix* the hierarchy, you could:
1. Use `create_heading_dict` to parse the structure
2. Analyze the dictionary to determine what the *correct* levels should be
3. Reconstruct the markdown with corrected heading levels

Would you like to explore how to add a "fix hierarchy" function that works with this module? Or would you prefer to use an LLM approach with the structured data from `create_heading_dict`?

Not sure using `create_heading_dict` would help. However leveraging the way the markdown is sanitized (for python) code is useful I think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're absolutely right! The key insight is the sanitization step - removing fenced code blocks before detecting headings to avoid false positives.

So for your LLM-based heading fixer, you could:
1. Strip out fenced blocks temporarily (like `re.sub(r'```[\s\S]*?```', '', text)`)
2. Extract headings from the sanitized text
3. Send those headings to the LLM to fix the hierarchy
4. Apply the corrections back to the original markdown

Does this approach sound like what you're thinking? Would you like to start building this, or discuss the prompt design for the LLM part?

In [ ]:
re.sub(r'```[\s\S]*?```', '', md)[:1000]

'# Deep Residual Learning for Image Recognition \n\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\\{kahe, v-xiangz, v-shren, jiansun\\}@microsoft.com\n\n\n#### Abstract\n\nDeeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers- $8 \\times$ deeper than VGG nets [41] but still having lower complexity. An ensemble of these residual nets achieves $3.57 \\%$ error on the ImageNet test set. This result won the 1st place on the ILSVRC 2015 classification

In [ ]:
def get_hdgs(md):
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return findall(r'^#{1,6} .+$', md, re.MULTILINE)



In [ ]:
hdgs = get_hdgs(md)
hdgs

['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']